# Curriculum 01 · Lab 6 — Chunk-Size Sweep

**Goal:** Sweep `chunk_size` and measure the **retrieval** impact — the classic
chunk-size tradeoff. Small chunks are precise but fragment a passage across many
vectors; large chunks are coherent but coarse (a whole passage may collapse into
one chunk, dragging neighbouring topics along).

```
Splitter    : RecursiveCharacterTextSplitter (chunk_size in {100, 200, 400, 800}, overlap=50)
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Vector DB   : FAISS (in-memory, fresh index per chunk size)
Retriever   : Similarity (Top-5), self-recall@5
Data        : rag-mini-wikipedia (first 1000 passages)
```

**Protocol — self-recall:** rag-mini-wikipedia has no qrels and its
`test.parquet` answers are yes/no, so we use a passage-sourced probe instead:
take ~50 random passages, derive a query from each (its first sentence),
retrieve top-5 chunks, and mark *"source passage found"* when any retrieved
chunk belongs to that passage. Each chunk size gets a **fresh** split + embed +
FAISS index so the comparison is clean.

This is the final lab of track 01-chunking (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Silences the model-download progress bars and the FAISS deprecation
notice, imports every dependency (pandas, tiktoken, LangChain document +
splitter), and puts the repo-root component library on `sys.path` so this
notebook reuses `embeddings/bge.py` and `vectordb/faiss.py` exactly like the
lab script.

**WHY:** Everything embeds **locally** with BGE via `sentence-transformers` —
no API embeddings anywhere. The env var `HF_HUB_DISABLE_PROGRESS_BARS` must be
set **before any third-party import** (`langchain_text_splitters` already pulls
in `huggingface_hub`, which reads the flag at import time), and the FAISS module
logger is set to `ERROR` so the deprecation notice for precomputed embeddings
doesn't bury the sweep table.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script,
`python curriculum/01-chunking/06-chunk-size-sweep.py`) or from the notebook's
own folder (the Jupyter default) — and `cd`s into it so every path stays
repo-relative.

**WHAT TO EXPECT:** no output — just a clean import. The model itself is loaded
lazily later.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings
#   faiss-cpu             -> local FAISS index
#   tiktoken              -> token counting for avg tokens/chunk
#   pandas                -> reads the passages.parquet corpus
%pip install sentence-transformers faiss-cpu tiktoken pandas


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import logging
import os
import random
import sys
import time
from pathlib import Path

# Keep the sweep table the star of the output: silence the "Loading weights"
# progress bar (transformers honors HF_HUB_DISABLE_PROGRESS_BARS) and the
# deprecation notice langchain_community.FAISS logs for precomputed
# embeddings (a logging call, so set the module logger to ERROR). The env var
# must be set before any third-party import — langchain_text_splitters already
# pulls in huggingface_hub, which reads the flag at import time.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
logging.getLogger("langchain_community.vectorstores.faiss").setLevel(logging.ERROR)

import pandas as pd
import tiktoken
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from embeddings.bge import BGEEmbedding  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402

## 1 · Configuration — the experiment's knobs

**WHAT:** The module-level constants that define the experiment: which corpus
slice to index, which chunk sizes to sweep, the overlap, and the self-recall
probe setup.

**WHY:** These are the knobs you tweak to re-run the experiment. `RANDOM_SEED`
is fixed so the probe set is reproducible; `CORPUS_SUBSET = 1000` keeps CPU
embedding time reasonable (the full corpus is 3200 passages).


In [3]:
# --- module-level constants: tweak these to re-run the experiment -----------
CORPUS_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
CORPUS_SUBSET = 1000  # first N passages — keeps CPU embedding time reasonable
CHUNK_SIZES = [100, 200, 400, 800]
OVERLAP = 50
PROBE_QUERIES = 50
TOP_K = 5
RANDOM_SEED = 42  # fixed seed so the probe set is reproducible
MODEL_NAME = "BAAI/bge-base-en-v1.5"

## 2 · Embedder — local BGE, with the repo's contract

**WHAT:** A BGE embedder exposing the repo's `embed_query` / `embed_documents`
contract, plus a factory that picks the implementation.

**WHY:** `langchain-huggingface` 1.x **dropped** `HuggingFaceBgeEmbeddings`, so
the repo's `BGEEmbedding` (which wraps it) can't be used here. `make_embedder()`
tries the repo class first and falls back to a direct `sentence-transformers`
wrapper with the same lazy-build + normalized-embedding semantics. Either way
the model is `BAAI/bge-base-en-v1.5`, cached locally — no download on this run.


In [4]:
class _SentenceTransformerEmbedder:
    """BGE embedder with the repo's ``embed_query``/``embed_documents`` contract.

    Stand-in for ``BGEEmbedding`` when the installed langchain-huggingface no
    longer exports ``HuggingFaceBgeEmbeddings`` (removed in 1.x). Keeps the
    same lazy-build + normalized-embedding semantics.
    """

    def __init__(self, model_name: str = MODEL_NAME):
        self.model_name = model_name
        self._model = None

    def _get_model(self):
        if self._model is None:
            from sentence_transformers import SentenceTransformer

            self._model = SentenceTransformer(self.model_name)
        return self._model

    def embed_query(self, text: str) -> list[float]:
        return (
            self._get_model()
            .encode(text, normalize_embeddings=True, show_progress_bar=False)
            .tolist()
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return (
            self._get_model()
            .encode(
                texts, normalize_embeddings=True, show_progress_bar=False
            )
            .tolist()
        )


def make_embedder():
    """Return a BGE embedder exposing ``embed_query``/``embed_documents``.

    Prefers the repo's ``BGEEmbedding``; falls back to a direct
    sentence-transformers wrapper when the installed langchain-huggingface
    can't provide ``HuggingFaceBgeEmbeddings``.
    """
    try:
        from langchain_huggingface import HuggingFaceBgeEmbeddings  # noqa: F401

        return BGEEmbedding(model_name=MODEL_NAME)
    except (ImportError, AttributeError) as exc:
        print(
            f"note: repo BGEEmbedding not usable here ({exc}); "
            "falling back to sentence-transformers directly"
        )
        return _SentenceTransformerEmbedder(model_name=MODEL_NAME)

## 3 · Helpers — load passages, derive probe queries

**WHAT:** Three small functions: `load_passages` reads the first `limit`
passages from the parquet corpus as `Document`s carrying a `passage_id`;
`derive_query` turns a passage into a probe query (first ~40 words, trimmed to
a complete sentence); `preview` truncates text for printing so we never dump
full chunk contents.

**WHY:** The `passage_id` metadata is what makes self-recall measurable — a
probe "hits" when any retrieved chunk carries the source passage's id.


In [5]:
def load_passages(path: Path, limit: int) -> list[Document]:
    """Load the first ``limit`` passages as Documents carrying a ``passage_id``."""
    df = pd.read_parquet(path)
    return [
        Document(page_content=text, metadata={"passage_id": i})
        for i, text in enumerate(df["passage"].head(limit).tolist())
    ]


def derive_query(text: str, max_words: int = 40) -> str:
    """Derive a probe query from a passage: first ~40 words, trimmed to a sentence.

    Cuts at the last sentence-ending punctuation inside the first ``max_words``
    so the query is a complete, self-contained sentence.
    """
    first = " ".join(text.split()[:max_words])
    cut = max((first.rfind(p) for p in ".!?"), default=-1)
    if cut > 0:
        first = first[: cut + 1]
    return first.strip() or text[:200].strip()


def preview(text: str, limit: int = 100) -> str:
    """Truncate text for printing (never dump full chunk contents)."""
    return text[:limit] + ("..." if len(text) > limit else "")

## 4 · One sweep pass — split → embed → index → probe

**WHAT:** `measure_chunk_size` runs one full pipeline for a single chunk size
and returns one table row: split the passages, embed every chunk with BGE,
build a **fresh** FAISS index, then probe it with the 50 derived queries.

**WHY:** Each chunk size gets a clean, independent index so the comparison is
fair. The row reports `n_chunks` (granularity), `avg_tokens` (chunk size in
tokens, via `tiktoken`), `recall` (self-recall@5 — fraction of probes whose
source passage appears in the top-5 hits), and `latency_ms` (mean query time).


In [6]:
def measure_chunk_size(
    size: int,
    passages: list[Document],
    probes: list[tuple[int, str]],
    embedder,
    enc,
) -> dict:
    """Split, embed, index and probe one chunk size; return one table row."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=OVERLAP
    )
    chunks = splitter.split_documents(passages)
    vectors = [list(v) for v in embedder.embed_documents(
        [c.page_content for c in chunks]
    )]

    store = FAISSVectorStore()
    store.add(chunks, embeddings=vectors)  # fresh index per chunk size

    avg_tokens = (
        sum(len(enc.encode(c.page_content)) for c in chunks) / len(chunks)
        if chunks
        else 0.0
    )

    found = 0
    latencies_ms: list[float] = []
    for passage_id, query in probes:
        query_vec = embedder.embed_query(query)
        t0 = time.perf_counter()
        hits = store.query(query_vec, top_k=TOP_K)
        latencies_ms.append((time.perf_counter() - t0) * 1000)
        if any(h.metadata.get("passage_id") == passage_id for h in hits):
            found += 1

    return {
        "chunk_size": size,
        "n_chunks": len(chunks),
        "avg_tokens": avg_tokens,
        "recall": found / len(probes),
        "latency_ms": sum(latencies_ms) / len(latencies_ms),
    }

## 5 · Run — setup printout, load the corpus subset, derive probes

**WHAT:** The first three steps of the experiment: print what we are sweeping,
load the 1000-passage subset, and derive the 50 probe queries from random
passages (fixed seed 42).

**WHY:** The setup printout makes the run self-describing; the probe derivation
is the self-recall protocol — each query is cut verbatim from a passage opening,
so a perfect index should find its source passage trivially.


In [7]:
# --- 1. setup: what we are sweeping and how we measure it ---------------
print("Chunk-size sweep — retrieval quality vs chunk size")
print(f"  corpus      : {CORPUS_PATH} (subset: first {CORPUS_SUBSET} passages)")
print(f"  embedder    : {MODEL_NAME} (local BGE, CPU)")
print(f"  chunk sizes : {CHUNK_SIZES}  overlap={OVERLAP}")
print(f"  probe       : {PROBE_QUERIES} passage-derived queries, "
      f"top-{TOP_K}, self-recall@5, seed={RANDOM_SEED}")
print("  (first run downloads the BGE model, ~440MB)\n")

# --- 2. load: a subset of passages as documents -------------------------
passages = load_passages(CORPUS_PATH, CORPUS_SUBSET)
avg_chars = sum(len(p.page_content) for p in passages) / len(passages)
print(f"Loaded {len(passages)} passages "
      f"(avg {avg_chars:.0f} chars/passage)")

# --- 3. probes: passage-derived queries (self-recall protocol) ----------
rng = random.Random(RANDOM_SEED)
probe_ids = rng.sample(range(len(passages)), PROBE_QUERIES)
probes = [(pid, derive_query(passages[pid].page_content)) for pid in probe_ids]
print(f"Derived {len(probes)} probe queries from random passages, e.g.:")
for pid, query in probes[:2]:
    print(f"  [{pid}] {preview(query)!r}")

Chunk-size sweep — retrieval quality vs chunk size
  corpus      : Data/corpus/rag-mini-wikipedia/passages.parquet (subset: first 1000 passages)
  embedder    : BAAI/bge-base-en-v1.5 (local BGE, CPU)
  chunk sizes : [100, 200, 400, 800]  overlap=50
  probe       : 50 passage-derived queries, top-5, self-recall@5, seed=42
  (first run downloads the BGE model, ~440MB)

Loaded 1000 passages (avg 411 chars/passage)
Derived 50 probe queries from random passages, e.g.:
  [654] 'Today, the Declaration of Independence is remembered as the great revolutionary act, but Adams and m...'
  [114] 'In 1848, as a result of representations by the Prince Consort, Michael Faraday was awarded a grace a...'


## 6 · Sweep — fresh split + embed + index per chunk size

**WHAT:** Build the embedder and tokenizer, then loop over the four chunk sizes
calling `measure_chunk_size` and printing one table row per size.

**WHY:** This is the experiment itself. Each row is a full pipeline run, so the
whole sweep takes a few minutes on CPU (the BGE model is cached — no download).
Watch the columns: `n_chunks` and `avg_tokens` grow/shrink with `chunk_size`,
while `self_recall@5` should stay at 1.000 everywhere (near-duplicate probes).


In [8]:
# --- 4. sweep: fresh split + embed + index per chunk size ---------------
embedder = make_embedder()
enc = tiktoken.encoding_for_model("gpt-4")
print("\nSweeping chunk sizes (fresh index per size)...\n")
header = (
    f"{'chunk_size':>10} | {'n_chunks':>9} | {'avg_tokens':>10} | "
    f"{'self_recall@5':>12} | {'mean_latency_ms':>15}"
)
print(header)
print("-" * len(header))
rows = []
for size in CHUNK_SIZES:
    row = measure_chunk_size(size, passages, probes, embedder, enc)
    rows.append(row)
    print(
        f"{row['chunk_size']:>10} | {row['n_chunks']:>9} | "
        f"{row['avg_tokens']:>10.1f} | {row['recall']:>12.3f} | "
        f"{row['latency_ms']:>15.2f}"
    )

note: repo BGEEmbedding not usable here (cannot import name 'HuggingFaceBgeEmbeddings' from 'langchain_huggingface' (/home/magus/.pyenv/versions/3.12.7/envs/magus/lib/python3.12/site-packages/langchain_huggingface/__init__.py)); falling back to sentence-transformers directly

Sweeping chunk sizes (fresh index per size)...

chunk_size |  n_chunks | avg_tokens | self_recall@5 | mean_latency_ms
---------------------------------------------------------------------


       100 |      7829 |       20.6 |        1.000 |            2.55


       200 |      2998 |       37.0 |        1.000 |            0.83


       400 |      1658 |       58.5 |        1.000 |            0.43


       800 |      1158 |       79.3 |        1.000 |            0.34


## 7 · Takeaway — what the table teaches

**WHAT:** Read the first and last rows and print the lesson the sweep teaches.

**WHY:** The self-recall column is deliberately flat — the real costs of chunk
size show up in `n_chunks`, `avg_tokens` and `latency_ms`. Small chunks mean
many vectors (more memory, more candidates per query, facts torn across
chunks); large chunks mean few fat vectors (fast and coherent, but coarse —
neighbouring topics get dragged in). The sweet spot is the smallest chunk size
that still keeps each passage's facts inside one retrievable unit.


In [9]:
# --- 5. takeaway: what the table teaches --------------------------------
small, large = rows[0], rows[-1]
print("\n--- takeaway: chunk size trades granularity against coherence ---")
print(
    f"chunk_size {small['chunk_size']} -> {large['chunk_size']}: "
    f"{small['n_chunks']} -> {large['n_chunks']} chunks "
    f"({small['n_chunks'] / large['n_chunks']:.1f}x fewer), "
    f"avg {small['avg_tokens']:.0f} -> {large['avg_tokens']:.0f} "
    "tokens/chunk"
)
print(
    "self-recall@5 stays high at every size because the probes are cut "
    "verbatim from passage openings — near-duplicate queries that any "
    "index finds. The real costs show in the other columns:"
)
print(
    "  * small chunks -> many vectors: more index memory, more candidates "
    "per query (latency), and a mid-passage fact gets torn across chunks, "
    "so retrieval returns pieces instead of the answer."
)
print(
    "  * large chunks -> few, fat vectors: fast and coherent, but coarse — "
    "a whole passage may be one chunk, so a query pulls in neighbouring "
    "topics and the context window fills with irrelevant text."
)
print(
    "The sweet spot is the smallest chunk size that still keeps each "
    "passage's facts inside one retrievable unit."
)


--- takeaway: chunk size trades granularity against coherence ---
chunk_size 100 -> 800: 7829 -> 1158 chunks (6.8x fewer), avg 21 -> 79 tokens/chunk
self-recall@5 stays high at every size because the probes are cut verbatim from passage openings — near-duplicate queries that any index finds. The real costs show in the other columns:
  * small chunks -> many vectors: more index memory, more candidates per query (latency), and a mid-passage fact gets torn across chunks, so retrieval returns pieces instead of the answer.
  * large chunks -> few, fat vectors: fast and coherent, but coarse — a whole passage may be one chunk, so a query pulls in neighbouring topics and the context window fills with irrelevant text.
The sweet spot is the smallest chunk size that still keeps each passage's facts inside one retrievable unit.
